<a href="https://colab.research.google.com/github/commertech-official/devops-start/blob/main/flow_coupling_and_moment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider
import ipywidgets as widgets

# ============================================================
# ПАРАМЕТРЫ ДВИГАТЕЛЯ
# ============================================================
p = 2          # число пар полюсов
R_s = 5.0      # сопротивление статора, Ом
R_r = 3.0      # сопротивление ротора, Ом
L_s = 0.3      # индуктивность статора, Гн
L_r = 0.3      # индуктивность ротора, Гн
L_m = 0.28     # взаимная индуктивность, Гн

# ============================================================
# РАСЧЁТ ПОТОКОСЦЕПЛЕНИЙ
# ============================================================
def calculate_fluxes(I_sd, I_sq, I_rd=0, I_rq=0):
    """
    Расчёт потокосцеплений статора и ротора.

    Ψ_sd = L_s * I_sd + L_m * I_rd
    Ψ_sq = L_s * I_sq + L_m * I_rq
    Ψ_rd = L_r * I_rd + L_m * I_sd
    Ψ_rq = L_r * I_rq + L_m * I_sq
    """
    Psi_sd = L_s * I_sd + L_m * I_rd
    Psi_sq = L_s * I_sq + L_m * I_rq
    Psi_rd = L_r * I_rd + L_m * I_sd
    Psi_rq = L_r * I_rq + L_m * I_sq
    return Psi_sd, Psi_sq, Psi_rd, Psi_rq

def calculate_torque(Psi_rd, Psi_rq, I_sd, I_sq):
    """
    M = (3/2) * p * (Ψ_rd * I_sq - Ψ_rq * I_sd)
    """
    return 1.5 * p * (Psi_rd * I_sq - Psi_rq * I_sd)

# ============================================================
# ГРАФИК 1: Зависимость потокосцеплений от I_sd и I_sq
# ============================================================
def plot_fluxes(I_sd=5.0, I_sq=5.0):
    # Для упрощения: токи ротора в установившемся режиме при Ψ_rq=0
    # I_rd = 0, I_rq = -(L_m/L_r) * I_sq
    I_rd = 0
    I_rq = -(L_m / L_r) * I_sq

    Psi_sd, Psi_sq, Psi_rd, Psi_rq = calculate_fluxes(I_sd, I_sq, I_rd, I_rq)
    M = calculate_torque(Psi_rd, Psi_rq, I_sd, I_sq)

    fig, axes = plt.subplots(2, 2, figsize=(14, 10))

    # --- График 1: Потокосцепления статора ---
    labels_s = ['Ψ_sd\n(поле)', 'Ψ_sq\n(момент)']
    values_s = [Psi_sd, Psi_sq]
    colors_s = ['#d62728', '#1f77b4']
    bars1 = axes[0, 0].bar(labels_s, values_s, color=colors_s, edgecolor='black', linewidth=1.5)
    axes[0, 0].set_ylabel('Потокосцепление, Вб')
    axes[0, 0].set_title(f'Потокосцепления статора\nI_sd = {I_sd:.1f} А, I_sq = {I_sq:.1f} А')
    axes[0, 0].axhline(y=0, color='black', linewidth=0.8)
    axes[0, 0].grid(True, alpha=0.3, axis='y')
    for bar, val in zip(bars1, values_s):
        axes[0, 0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                        f'{val:.3f}', ha='center', va='bottom', fontweight='bold')

    # --- График 2: Потокосцепления ротора ---
    labels_r = ['Ψ_rd\n(поле)', 'Ψ_rq\n(момент)']
    values_r = [Psi_rd, Psi_rq]
    colors_r = ['#2ca02c', '#ff7f0e']
    bars2 = axes[0, 1].bar(labels_r, values_r, color=colors_r, edgecolor='black', linewidth=1.5)
    axes[0, 1].set_ylabel('Потокосцепление, Вб')
    axes[0, 1].set_title(f'Потокосцепления ротора\nI_sd = {I_sd:.1f} А, I_sq = {I_sq:.1f} А')
    axes[0, 1].axhline(y=0, color='black', linewidth=0.8)
    axes[0, 1].grid(True, alpha=0.3, axis='y')
    for bar, val in zip(bars2, values_r):
        axes[0, 1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                        f'{val:.3f}', ha='center', va='bottom', fontweight='bold')

    # --- График 3: Момент ---
    axes[1, 0].bar(['M'], [M], color='#9467bd', edgecolor='black', linewidth=1.5, width=0.5)
    axes[1, 0].set_ylabel('Момент, Н·м')
    axes[1, 0].set_title(f'Электромагнитный момент\nM = (3/2)·p·(Ψ_rd·I_sq - Ψ_rq·I_sd)')
    axes[1, 0].axhline(y=0, color='black', linewidth=0.8)
    axes[1, 0].grid(True, alpha=0.3, axis='y')
    axes[1, 0].text(0, M + 0.01, f'{M:.4f} Н·м', ha='center', va='bottom', fontweight='bold')

    # --- График 4: Векторная диаграмма ---
    ax = axes[1, 1]

    # Вектор потокосцепления ротора
    ax.arrow(0, 0, Psi_rd, Psi_rq, head_width=0.02, head_length=0.03,
             fc='green', ec='green', linewidth=3, label=f'Ψ_r (ротор)')

    # Вектор потокосцепления статора
    ax.arrow(0, 0, Psi_sd, Psi_sq, head_width=0.02, head_length=0.03,
             fc='red', ec='red', linewidth=3, label=f'Ψ_s (статор)')

    # Вектор тока статора
    scale_I = 0.05
    ax.arrow(0, 0, I_sd*scale_I, I_sq*scale_I, head_width=0.02, head_length=0.03,
             fc='blue', ec='blue', linewidth=3, linestyle='--', label=f'I_s (статор)')

    ax.set_xlabel('Ось d (намагничивание)')
    ax.set_ylabel('Ось q (момент)')
    ax.set_title('Векторная диаграмма в системе d-q')
    ax.grid(True, alpha=0.3)
    ax.axhline(y=0, color='black', linewidth=0.8)
    ax.axvline(x=0, color='black', linewidth=0.8)
    ax.legend(loc='upper right')
    ax.set_aspect('equal')

    # Установка пределов
    max_val = max(abs(Psi_sd), abs(Psi_sq), abs(Psi_rd), abs(Psi_rq), abs(I_sd*scale_I), abs(I_sq*scale_I)) * 1.3
    ax.set_xlim(-max_val, max_val)
    ax.set_ylim(-max_val, max_val)

    plt.tight_layout()
    plt.show()

    # Вывод численных результатов
    print("="*60)
    print(f"ВХОДНЫЕ ДАННЫЕ:")
    print(f"  I_sd = {I_sd:.2f} А (ток намагничивания)")
    print(f"  I_sq = {I_sq:.2f} А (ток момента)")
    print(f"  I_rd = {I_rd:.2f} А, I_rq = {I_rq:.2f} А")
    print("-"*60)
    print(f"ПОТОКОСЦЕПЛЕНИЯ СТАТОРА:")
    print(f"  Ψ_sd = {Psi_sd:.4f} Вб")
    print(f"  Ψ_sq = {Psi_sq:.4f} Вб")
    print(f"ПОТОКОСЦЕПЛЕНИЯ РОТОРА:")
    print(f"  Ψ_rd = {Psi_rd:.4f} Вб  ← определяет поле")
    print(f"  Ψ_rq = {Psi_rq:.4f} Вб  ← должен быть 0 при FOC")
    print("-"*60)
    print(f"МОМЕНТ:")
    print(f"  M = {M:.4f} Н·м")
    print("="*60)

# ============================================================
# ЗАПУСК ИНТЕРАКТИВНОГО ГРАФИКА
# ============================================================
interact(plot_fluxes,
         I_sd=FloatSlider(min=0, max=10, step=0.5, value=5.0,
                          description='Id (А):'),
         I_sq=FloatSlider(min=-10, max=10, step=0.5, value=5.0,
                          description='Iq (А):'))

interactive(children=(FloatSlider(value=5.0, description='Id (А):', max=10.0, step=0.5), FloatSlider(value=5.0…

<function __main__.plot_fluxes(I_sd=5.0, I_sq=5.0)>